Live webcam demo for the trained emotion model. Streams frames from the browser into Colab via a JS bridge (Colab cannot open the local webcam directly), runs an OpenCV Haar cascade to find faces, classifies each face with the trained MobileNetV2 emotion model, and overlays the predicted emotion label and confidence on the live video. Click the video to stop. No training happens here; the model file is loaded from Drive.

Mount Drive, clone the repo, install dependencies, and verify the trained model file is available at the expected path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d COS30082-Facial-Recognition ] || git clone https://github.com/jeromeliao03/COS30082-Facial-Recognition.git
%cd /content/COS30082-Facial-Recognition
!git fetch origin && git checkout emotion-branch && git pull --ff-only

In [ ]:
!pip install -q tensorflow numpy Pillow opencv-python

In [ ]:
import sys, pathlib
REPO_ROOT = '/content/COS30082-Facial-Recognition'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

MODEL_PATH = '/content/drive/MyDrive/MLGroup/emotion/models/mobilenetv2_best.keras'
assert pathlib.Path(MODEL_PATH).exists(), f'Model file not found at {MODEL_PATH}'
print('Model file present:', MODEL_PATH)

Build the predictor once. The wrapper loads the .keras file, handles resizing to 128x128, MobileNetV2 preprocessing, and softmax. input_is_bgr is True because OpenCV gives BGR frames.

In [ ]:
from emotion.src.predict import EmotionPredictor

predictor = EmotionPredictor(MODEL_PATH, input_is_bgr=True)
print('Predictor ready.')

Load a Haar cascade face detector. Ships with OpenCV, no extra download needed. Fast enough for live video, less accurate than MediaPipe or MTCNN but adequate for this demo. The cascade returns bounding boxes; we crop each face and pass it to the emotion predictor.

In [ ]:
import cv2

cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)
assert not face_cascade.empty(), 'Haar cascade failed to load'
print('Face detector ready.')

Per-frame processing: detect faces in the frame, run the emotion predictor on each crop, draw a bounding box and an emotion label with confidence. Returns the annotated frame plus the list of detected emotions for the status line.

In [ ]:
import numpy as np

EMOTION_COLOURS = {
    'angry':    (0, 0, 200),
    'disgust':  (0, 102, 0),
    'fear':     (128, 0, 128),
    'happy':    (0, 200, 200),
    'neutral':  (180, 180, 180),
    'sad':      (200, 100, 0),
    'surprise': (0, 200, 0),
}

def process_frame(frame_bgr):
    """Detect faces, classify emotion, annotate. Returns (annotated_frame, labels)."""
    labels_seen = []
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.2, minNeighbors=5, minSize=(60, 60)
    )

    for (x, y, w, h) in faces:
        crop = frame_bgr[y:y+h, x:x+w]
        if crop.size == 0:
            continue
        try:
            result = predictor.predict(crop)
        except Exception:
            continue
        label = result['label']
        conf = result['confidence']
        labels_seen.append(f'{label} ({conf:.0%})')

        colour = EMOTION_COLOURS.get(label, (255, 255, 255))
        cv2.rectangle(frame_bgr, (x, y), (x+w, y+h), colour, 2)
        cv2.putText(
            frame_bgr, f'{label} {conf:.0%}', (x, max(y - 8, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, colour, 2,
        )

    return frame_bgr, labels_seen

Colab webcam bridge. Same JS pattern as lab07. The browser captures webcam frames and pushes them to Python; the annotated frame is composited back as a transparent PNG overlay aligned over the live video element. Click the video to stop.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io, PIL.Image

def _video_stream_js():
    js = Javascript('''
      var video, stream, captureCanvas, imgElement, labelElement, div;
      var pendingResolve = null;
      var shutdown = false;

      function removeDom() {
        stream.getVideoTracks()[0].stop();
        video.remove(); div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }
      function onAnimationFrame() {
        if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
        if (pendingResolve) {
          var result = '';
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
            result = captureCanvas.toDataURL('image/jpeg', 0.8);
          }
          var lp = pendingResolve; pendingResolve = null; lp(result);
        }
      }
      async function createDom() {
        if (div !== null && div !== undefined) { return stream; }
        div = document.createElement('div');
        div.style.border = '2px solid black'; div.style.padding = '3px';
        div.style.width = '100%'; div.style.maxWidth = '660px';
        document.body.appendChild(div);
        const modelOut = document.createElement('div');
        modelOut.innerHTML = '<span>Status: </span>';
        labelElement = document.createElement('span');
        labelElement.innerText = 'Starting...';
        labelElement.style.fontWeight = 'bold';
        modelOut.appendChild(labelElement); div.appendChild(modelOut);
        video = document.createElement('video');
        video.style.display = 'block'; video.width = div.clientWidth - 6;
        video.setAttribute('playsinline', '');
        video.onclick = () => { shutdown = true; };
        stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'user'}});
        div.appendChild(video);
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute'; imgElement.style.zIndex = 1;
        imgElement.onclick = () => { shutdown = true; };
        div.appendChild(imgElement);
        const instruction = document.createElement('div');
        instruction.innerHTML =
          '<span style="color: red; font-weight: bold;">' +
          'Click the video to stop</span>';
        div.appendChild(instruction);
        instruction.onclick = () => { shutdown = true; };
        video.srcObject = stream; await video.play();
        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640; captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);
        return stream;
      }
      async function stream_frame(label, imgData) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await createDom();
        if (label != '') { labelElement.innerHTML = label; }
        if (imgData != '') {
          var videoRect = video.getClientRects()[0];
          imgElement.style.top = videoRect.top + 'px';
          imgElement.style.left = videoRect.left + 'px';
          imgElement.style.width = videoRect.width + 'px';
          imgElement.style.height = videoRect.height + 'px';
          imgElement.src = imgData;
        }
        var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''')
    display(js)

def video_frame(label, bbox_data):
    return eval_js(f'stream_frame("{label}", "{bbox_data}")')

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    arr = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)

def bbox_to_bytes(bbox_array):
    pil = PIL.Image.fromarray(bbox_array, 'RGBA')
    buf = io.BytesIO(); pil.save(buf, format='png')
    return 'data:image/png;base64,{}'.format(b64encode(buf.getvalue()).decode('utf-8'))

Main loop. Captures a webcam frame, runs face detection and emotion classification, composites the bounding boxes and labels onto a transparent overlay that sits on top of the live video. Click the video preview to stop.

In [ ]:
_video_stream_js()
label_html = 'Starting...'
bbox = ''
frame_count = 0

while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break
    frame = js_to_image(js_reply['img'])

    overlay = np.zeros((frame.shape[0], frame.shape[1], 4), dtype=np.uint8)
    annotated, labels = process_frame(frame.copy())

    diff = cv2.absdiff(annotated, frame).sum(axis=2) > 0
    overlay[diff, :3] = annotated[diff]
    overlay[diff, 3] = 255

    bbox = bbox_to_bytes(overlay)
    frame_count += 1
    summary = ', '.join(labels) if labels else 'no face detected'
    label_html = f'Frame {frame_count} | {summary}'

print('Stopped.')